# Capítulo III – Análisis de datos y estadística descriptiva con R
## Análisis Cuantitativo con R: Matemáticas, Estadística y Econometría

**Autores del libro:** Daniel Liviano Solís · Maria Pujol Jover
**Editorial:** UOC · Primera edición digital: junio 2017
**Fuente:** Capítulo III, págs. 127-150

---

### Objetivos de aprendizaje
Al finalizar este notebook serás capaz de:
1. Importar datos desde distintos formatos (`.xlsx`, `.txt`, `.csv`, SPSS, SAS, Stata).
2. Explorar un `data.frame` con `head()`, `tail()`, `dim()` y `summary()`.
3. Filtrar y ordenar observaciones con `subset()` y `order()`.
4. Crear nuevas variables mediante operaciones vectorizadas.
5. Calcular estadísticos descriptivos con funciones propias y `apply()`/`tapply()`.
6. Calcular correlaciones y covarianzas entre variables.
7. Representar gráficamente datos: dispersión, histograma, densidad y diagrama de caja.

### Datos utilizados
El capítulo trabaja con datos del **Instituto de Estadística de Catalunya (Idescat)**, año 2009,
para 941 municipios catalanes:

| Variable | Descripción |
|----------|-------------|
| `mun`    | Código del municipio |
| `pob`    | Población total |
| `tra`    | Población activa |
| `mig`    | Población inmigrante |
| `edad`   | Media de edad de la población |
| `costa`  | Variable dicotómica (Sí/No): si el municipio está en la costa |

### Cómo usar este notebook
Cada apartado sigue el **mismo número de sección del libro** (p. ej. `4.3` = función `estad.basic`).
El libro trabaja con un archivo externo `Datos.xlsx`; para que el notebook sea ejecutable de
forma autónoma (p. ej. en Google Colab), la celda siguiente genera un **dataset sintético** con
la misma estructura y estadísticos aproximados. El resto del código reproduce exactamente el
del libro operando sobre `base.datos`.

---
## Preparación: dataset sintético equivalente a `Datos.xlsx`

> ⚠️ Esta celda **no proviene del libro**: sustituye la importación real de `Datos.xlsx` (no distribuido) por datos sintéticos con la misma estructura, para que el resto del notebook —código original del libro— pueda ejecutarse sin depender de un archivo externo.

In [ ]:
# ---------------------------------------------------------------
# PREPARACION (no es codigo del libro): genera un dataset sintetico
# con la misma estructura que Datos.xlsx (941 municipios, Idescat 2009)
# ---------------------------------------------------------------
set.seed(42)   # semilla para reproducibilidad
n <- 941       # numero de municipios

# Crear data frame con la misma estructura que el original
base.datos <- data.frame(
  mun   = paste0("M", 1:n),                            # codigo municipio
  pob   = as.integer(rlnorm(n, meanlog = 7, sdlog = 1.5)), # poblacion total
  tra   = as.integer(rlnorm(n, meanlog = 6, sdlog = 1.5)), # poblacion activa
  mig   = as.integer(rlnorm(n, meanlog = 5, sdlog = 1.8)), # inmigrantes
  edad  = round(rnorm(n, mean = 42.79, sd = 4.5), 2),      # media de edad
  costa = factor(sample(c("No", "Si"), n,                  # variable factor
                        replace = TRUE, prob = c(0.926, 0.074)))
)

cat("Dataset sintetico creado:", nrow(base.datos), "filas x",
    ncol(base.datos), "columnas\n")

---
# 1. Motivación (pág. 127)

El objetivo de este capítulo es mostrar cómo analizar datos en R: importación desde
diferentes formatos, creación y manipulación de variables, matrices y bases de datos,
cálculo de estadísticos, visualización y representación gráfica.

---
# 2. Importación de datos (pág. 127-129)

### Conceptos clave
Antes de importar datos es fundamental fijar el **directorio de trabajo** con `setwd()`.
Para archivos Excel se usa la librería `xlsx` y `read.xlsx()`; el segundo argumento indica
la hoja del documento. Alternativas según formato:
- Texto (`.txt`): `read.table()` / `read.delim()` / `read.delim2()`
- CSV (`.csv`): `read.csv()` (separador coma, `dec="."`)
- SPSS: `Hmisc::spss.get()` · SAS: `Hmisc::sasxport.get()` · Stata (`.dta`): `foreign::read.dta()`

In [ ]:
# ---------------------------------------------------------------
# 2a Fijar el directorio de trabajo (pag. 127-128)
# ---------------------------------------------------------------
setwd("C:/Directorio")

In [ ]:
# ---------------------------------------------------------------
# 2b Importar datos desde Excel: library(xlsx) + read.xlsx() (pag. 128)
# base.datos es un objeto data.frame; el 1 indica la hoja del libro Excel
# ---------------------------------------------------------------
library(xlsx)
base.datos <- read.xlsx("Datos.xlsx", 1)

# Nota: en este notebook se usa el dataset sintetico generado en la
# celda de preparacion, ya que Datos.xlsx no esta disponible.

---
# 3. Visualización y manejo de datos (pág. 129-134)

### 3.1. Primeras observaciones: `head()` (pág. 129)

`head()` muestra en consola las 6 primeras observaciones en filas, con las variables en columnas. Es la manera habitual de comprobar que los datos se han importado correctamente.

In [ ]:
# ---------------------------------------------------------------
# 3.1 head(): primeras observaciones (pag. 129)
# ---------------------------------------------------------------
head(base.datos)
#   mun   pob  tra  mig  edad costa
# 1  M1 11521 5712 1152 36.77    No
# 2  M2   257   17   12 43.54    No
# 3  M3  9397 1456  874 39.91    No
# 4  M4   311   27   23 41.23    No
# 5  M5  7949 1843  589 36.99    No
# 6  M6 14627 2487 1719 41.84    Si

### 3.2. Dimensión del objeto: `dim()` (pág. 130)

`dim()` aplicada a una base de datos muestra, en primer lugar, el número de filas (observaciones) y, en segundo lugar, el número de columnas (variables).

In [ ]:
# ---------------------------------------------------------------
# 3.2 dim(): filas x columnas (pag. 130)
# ---------------------------------------------------------------
dim(base.datos)
# [1] 941   6   -> 941 municipios, 6 variables

### 3.3. Resumen estadístico: `summary()` (pág. 130-131)

`summary()` muestra, para cada variable, el valor mínimo, máximo, la media y los tres cuartiles. Para variables factor muestra la frecuencia de cada categoría.

In [ ]:
# ---------------------------------------------------------------
# 3.3 summary(): resumen estadistico (pag. 130-131)
# ---------------------------------------------------------------
summary(base.datos)
#      mun          pob              tra              mig
#  M1     :  1   Min.   :    29   Min.   :    0   Min.   :    0
#  M10    :  1   1st Qu.:   334   1st Qu.:   39   1st Qu.:   23
#  M100   :  1   Median :   971   Median :  148   Median :   87
#  M101   :  1   Mean   :  7944   Mean   : 2461   Mean   : 1146
#  M102   :  1   3rd Qu.:  3726   3rd Qu.:  993   3rd Qu.:  536
#  M103   :  1   Max.   :1621537  Max.   :858437  Max.   :369514

### 3.4. Filtrado con `subset()` (pág. 131-132)

`subset()` crea subconjuntos del data frame aplicando condiciones lógicas sobre las filas y seleccionando columnas específicas con `select`.

In [ ]:
# ---------------------------------------------------------------
# 3.4a subset(): condicion simple (pag. 131)
# Municipios con edad media > 58 anios
# ---------------------------------------------------------------
subset(base.datos, edad > 58,
       select = c(mun, pob, edad))
#      mun pob      edad
# 819 M819  43  60.32558
# 916 M916 119  58.02521

In [ ]:
# ---------------------------------------------------------------
# 3.4b subset(): condicion doble con & (pag. 132)
# Municipios con edad < 40 Y poblacion < 200 habitantes
# El operador & exige que AMBAS condiciones se cumplan
# ---------------------------------------------------------------
subset(base.datos, edad < 40 & pob < 200,
       select = c(mun, pob, edad))
#      mun pob      edad
# 253 M253 163  39.71779
# 311 M311 152  38.71053
# 451 M451 141  38.64539
# 598 M598 152  37.86184

### 3.5. Ordenar observaciones: `order()` (pág. 132)

`order(variable)` devuelve los índices de posición que ordenarían la variable de forma **ascendente**; con signo negativo, `order(-variable)`, el orden es **descendente**.

In [ ]:
# ---------------------------------------------------------------
# 3.5 order(): ordenar por edad (pag. 132)
# tail() muestra los 6 municipios con mayor media de edad
# ---------------------------------------------------------------
tail(base.datos[order(edad), ])
#      mun  pob tra mig      edad costa
# 585 M585  109  32   3  54.94495    No
# 858 M858  159  22  12  55.11950    No
# 621 M621  138   3   2  56.65217    No
# 571 M571   63   7   3  56.87302    No
# 916 M916  119  15  20  58.02521    No
# 819 M819   43   5   0  60.32558    No

### 3.6. Acceso a variables con `$` y `attach()` (pág. 132-133)

El operador `$` accede a columnas individuales: `base.datos$pob`. `attach()` incrusta las variables del data frame en el entorno global para referenciarlas directamente por nombre.

In [ ]:
# ---------------------------------------------------------------
# 3.6a Acceso con $ (pag. 132)
# ---------------------------------------------------------------
mean(base.datos$pob)
# [1] 7944.123

In [ ]:
# ---------------------------------------------------------------
# 3.6b attach(): incrustar variables en el entorno global (pag. 133)
# ---------------------------------------------------------------
attach(base.datos)

# Ahora la media se calcula directamente con el nombre de la variable
mean(pob)
# [1] 7944.123

### 3.7. Creación de nuevas variables (pág. 133-134)

Gracias a la vectorización, crear indicadores a partir de variables existentes es inmediato.

Tasa de actividad = 100 · población activa / población total
Tasa de inmigración = 100 · población inmigrante / población total

In [ ]:
# ---------------------------------------------------------------
# 3.7a Crear tasas de actividad e inmigracion por municipio (pag. 133)
# La division es vectorizada: opera sobre los 941 municipios a la vez
# ---------------------------------------------------------------

# Tasa de actividad: porcentaje de poblacion activa sobre el total
t.act <- 100 * tra / pob

# Tasa de inmigracion: porcentaje de inmigrantes sobre el total
t.mig <- 100 * mig / pob

In [ ]:
# ---------------------------------------------------------------
# 3.7b Tasas agregadas para toda Catalunya (pag. 134)
# ---------------------------------------------------------------

# Tasa de actividad de Catalunya: ~= 32.8%
100 * sum(tra) / sum(pob)
# [1] 32.81848

# Tasa de inmigracion de Catalunya: ~= 15.9%
100 * sum(mig) / sum(pob)
# [1] 15.90919

---
# 4. Cálculo de estadísticos (pág. 135-140)

### 4.1. Matriz de indicadores con `cbind()` y `row.names()` (pág. 135)

`cbind()` une vectores por columnas creando una matriz. `row.names()` asigna nombres a las filas.

In [ ]:
# ---------------------------------------------------------------
# 4.1 Construir matriz de indicadores (pag. 135)
# ---------------------------------------------------------------

# Combinar los tres vectores por columnas
indic <- cbind(t.act, t.mig, edad)

# Asignar el codigo de municipio como nombre de cada fila
row.names(indic) <- mun

### 4.2. Resumen y `apply()` sobre la matriz de indicadores (pág. 135-136)

`apply(X, MARGIN, FUN)` aplica una función sobre filas (`MARGIN=1`) o columnas (`MARGIN=2`) de una matriz.

In [ ]:
# ---------------------------------------------------------------
# 4.2a summary() sobre la matriz indic (pag. 135)
# ---------------------------------------------------------------
summary(indic)
#      t.act             t.mig              edad
# Min.   : 0.000   Min.   : 0.000   Min.   :33.63
# 1st Qu.: 8.878   1st Qu.: 5.007   1st Qu.:39.37
# Median :15.494   Median : 9.127   Median :42.22
# Mean   :21.213   Mean   :10.442   Mean   :42.79
# 3rd Qu.:25.506   3rd Qu.:14.238   3rd Qu.:45.87
# Max.   :266.138  Max.   :50.896   Max.   :60.33

In [ ]:
# ---------------------------------------------------------------
# 4.2b apply(): media de cada columna (pag. 136)
# ---------------------------------------------------------------
apply(indic, 2, mean)
#     t.act     t.mig      edad
#  21.21259  10.44170  42.79377

### 4.3. Función personalizada de estadísticos: `estad.basic()` (pág. 136-137)

Función propia que calcula en una sola llamada: media, varianza, desviación estándar, mínimo, tres cuartiles y máximo, usando `cbind()`, `t()`, `colnames()` y `round()`.

> Esta función puede guardarse en un archivo y cargarse con `source()` en futuros análisis.

In [ ]:
# ---------------------------------------------------------------
# 4.3a Definicion de estad.basic() (pag. 136-137)
# Parametro: x (vector numerico)
# Retorna: matriz 1x8 con media, varianza, desv.est, min, Q1, Q2, Q3, max
# ---------------------------------------------------------------
estad.basic <- function(x) {
  # Combinar estadisticos por filas: cada uno es un escalar
  est <- cbind(mean(x), var(x), sd(x),
               t(quantile(x)))

  # Asignar nombres a cada columna de la matriz resultante
  colnames(est) <- c("media", "var",
                     "desv.est", "min", "Q1", "Q2", "Q3", "max")

  # Retornar el resultado redondeado a 2 decimales
  return(round(est, 2))
}

In [ ]:
# ---------------------------------------------------------------
# 4.3b Cargar la funcion desde un archivo externo (pag. 137)
# En entorno local podria cargarse asi; en este notebook ya esta
# definida en la celda anterior.
# ---------------------------------------------------------------
source("estad.basic.txt")

In [ ]:
# ---------------------------------------------------------------
# 4.3c Aplicar estad.basic() a la tasa de actividad (pag. 137)
# ---------------------------------------------------------------
estad.basic(t.act)
#      media    var desv.est min   Q1    Q2    Q3    max
# [1,] 21.21 484.02       22   0 8.88 15.49 25.51 266.14

### 4.4. Estadísticos para todas las variables: `apply()` + `estad.basic` (pág. 137-138)

Combinando `apply()` con la función personalizada `estad.basic`, se obtienen todos los estadísticos para las tres variables de `indic` en un único cálculo.

In [ ]:
# ---------------------------------------------------------------
# 4.4 apply(indic, 2, estad.basic): estadisticos de las 3 variables (pag. 137-138)
# ---------------------------------------------------------------
est.total <- apply(indic, 2, estad.basic)

# Asignar nombres de filas a los estadisticos
rownames(est.total) <- c("media", "var",
                         "desv.est", "min", "Q1", "Q2", "Q3", "max")

print(est.total)
#            t.act  t.mig   edad
# media      21.21  10.44  42.79
# var       484.02  54.49  18.65
# desv.est   22.00   7.38   4.32
# min         0.00   0.00  33.63
# Q1          8.88   5.01  39.37
# Q2         15.49   9.13  42.22
# Q3         25.51  14.24  45.87
# max       266.14  50.90  60.33

### 4.5. Variables categóricas: `class()`, `levels()` y `tapply()` (pág. 138-139)

Las variables cualitativas se representan como **factores**. `class()` indica el tipo del objeto, `levels()` muestra las categorías, y `tapply()` aplica una función a un vector agrupado por los niveles de un factor.

In [ ]:
# ---------------------------------------------------------------
# 4.5a class() y levels() sobre el factor costa (pag. 138)
# ---------------------------------------------------------------
class(costa)
# [1] "factor"

levels(costa)
# [1] "No" "Si"

In [ ]:
# ---------------------------------------------------------------
# 4.5b tapply(): tasa de inmigracion segun costa (pag. 138-139)
# ---------------------------------------------------------------
tapply(t.mig, costa, mean)
#     No       Si
#  9.691   19.770
# -> La tasa de inmigracion es mucho mayor en municipios de la costa

### 4.6. Correlaciones con `cor()` (pág. 139-140)

`cor()` calcula la matriz de correlaciones de Pearson entre todas las variables de una matriz. `print(..., digits=2)` limita los dígitos mostrados. `cov()` calcula la matriz de covarianzas.

In [ ]:
# ---------------------------------------------------------------
# 4.6 Matriz de correlaciones: cor() (pag. 139-140)
# ---------------------------------------------------------------
print(cor(indic), digits = 2)
#        t.act   t.mig   edad
# t.act 1.0000  0.0072 -0.2518
# t.mig 0.0072  1.0000 -0.2752
# edad -0.2518 -0.2752  1.0000
# -> edad correlaciona negativamente con t.mig (-0.28): municipios
#    con mas inmigrantes tienden a tener poblaciones mas jovenes

---
# 5. Representación gráfica (pág. 140-146)

R ofrece un amplio catálogo de gráficos:

| Descripción | Instrucción |
|-------------|-------------|
| Dispersión con puntos | `plot(x, type="p")` |
| Gráfico con línea | `plot(x, type="l")` |
| Líneas y puntos superpuestos | `plot(x, type="o")` |
| Líneas y puntos discontinuos | `plot(x, type="b")` |
| Escalera | `plot(x, type="s")` |
| Histograma | `hist(x)` |
| Densidad estimada | `plot(density(x))` |
| Gráfico de puntos | `dotchart(x)` |
| Gráfico de barras | `barplot(x)` |
| Gráfico circular | `pie(x)` |
| Diagrama de caja | `boxplot(x)` |

### 5.1. Diagrama de dispersión con recta ajustada (pág. 140-141)

`plot(x, y)` genera un diagrama de dispersión. `abline(lm(...))` superpone la recta de regresión lineal.

In [ ]:
# ---------------------------------------------------------------
# 5.1 Dispersion: tasa de inmigracion vs edad (pag. 140-141)
# ---------------------------------------------------------------

# Diagrama de dispersion: cada punto es un municipio catalan
plot(t.mig, edad)

# Superponer la recta de regresion lineal (edad explicada por t.mig)
abline(lm(edad ~ t.mig), lwd = 4)

### 5.2. Diagrama de dispersión con variables centradas (pág. 141-142)

Centrar las variables (restar la media) desplaza el origen al punto (x̄, ȳ). `abline(h=0)` y `abline(v=0)` añaden ejes de referencia en cero.

In [ ]:
# ---------------------------------------------------------------
# 5.2 Dispersion con variables centradas (pag. 141-142)
# ---------------------------------------------------------------

# Transformacion: xi - x_barra para cada observacion (vectorizacion)
plot(t.mig - mean(t.mig), edad - mean(edad))

# Ejes de referencia en cero
abline(h = 0)
abline(v = 0)

### 5.3. Histograma con `hist()` (pág. 142-143)

`hist()` visualiza la distribución de una variable continua. Aplicar `log()` antes de `hist()` es útil cuando la variable tiene distribución muy asimétrica (como la población).

In [ ]:
# ---------------------------------------------------------------
# 5.3 Histograma de la poblacion en escala logaritmica (pag. 142-143)
# ---------------------------------------------------------------
hist(log(pob),
     main = "Histograma de la poblacion (en logaritmos)",
     col = "grey")

### 5.4. Función de densidad estimada con `density()` y `polygon()` (pág. 143-144)

`density()` estima la densidad de probabilidad de forma no paramétrica (kernel). `polygon()` rellena el área bajo la curva.

In [ ]:
# ---------------------------------------------------------------
# 5.4 Densidad estimada de la edad (pag. 143-144)
# ---------------------------------------------------------------

# density() usa estimacion kernel no parametrica
den.edad <- density(edad)

# Dibujar la curva de densidad
plot(den.edad, main = "Densidad de la media de edad")

# Rellenar el area bajo la curva
polygon(den.edad, col = "grey", border = "black")

### 5.5. Diagrama de caja por grupos: `boxplot()` (pág. 144-146)

`boxplot(y ~ factor)` genera un diagrama de caja por cada nivel del factor. La caja muestra Q1, mediana y Q3; los puntos fuera de 1.5×IQR son valores atípicos.

In [ ]:
# ---------------------------------------------------------------
# 5.5 Boxplot de edad segun costa (pag. 144-146)
# ---------------------------------------------------------------
boxplot(edad ~ costa, col = "grey")

# Interpretacion esperada:
# Las poblaciones en la costa tienden a ser mas jovenes;
# los municipios del interior presentan mayor dispersion de edad.

---
## Conclusión

Este notebook cubre la totalidad del código R del **Capítulo III** de Liviano & Pujol (2017), siguiendo el mismo orden y numeración de secciones que el texto original.

| Sección del libro | Tema | Funciones clave |
|---|---|---|
| 1 | Motivación | — |
| 2 | Importación de datos | `setwd()`, `read.xlsx()`, `read.csv()`, `read.table()` |
| 3.1-3.3 | Exploración | `head()`, `tail()`, `dim()`, `summary()` |
| 3.4-3.5 | Filtrado y ordenación | `subset()`, `order()` |
| 3.6 | Acceso a variables | `$`, `attach()` |
| 3.7 | Creación de variables | operaciones vectorizadas |
| 4.1 | Matrices de indicadores | `cbind()`, `row.names()` |
| 4.2-4.4 | Estadísticos | `apply()`, funciones propias (`estad.basic`) |
| 4.5 | Factores | `class()`, `levels()`, `tapply()` |
| 4.6 | Correlación | `cor()`, `cov()` |
| 5 | Visualización | `plot()`, `hist()`, `density()`, `polygon()`, `boxplot()`, `abline()` |

### Cómo ejecutar
1. Ejecutar primero la celda de **preparación** (dataset sintético).
2. Abrir en **Jupyter Lab**, **VS Code** o **Google Colab** con kernel `R`.
3. Ejecutar el resto de celdas en orden: reutilizan `base.datos`, `t.act`, `t.mig`, `indic`.
4. Las celdas de la Sección 2 (importación real) y 4.3b (`source()`) están documentadas
   tal como aparecen en el libro, pero requieren archivos que no forman parte de este repositorio.

---
*Notebook estandarizado a partir del PDF oficial del capítulo (Cap03_Analisis_Datos_Estadistica_Descriptiva_R.pdf).*
*Referencia: Liviano Solís, D. & Pujol Jover, M. (2017). Análisis cuantitativo con R: matemáticas, estadística y econometría. Editorial UOC.*